# YOLO i RF-DETR - demo na zywo z kamery

Ten notebook uruchamia dwa osobne dema detekcji odpadow z wbudowanej kamery laptopa:

- YOLO z wagami `weights_yolo.pt`
- RF-DETR z wagami `weights_rfdetr.pt`

Oba modele sa traktowane jako modele wytrenowane na customowym datasecie `GARBAGE-CLASSIFICATION-3-2`. Nazwy klas sa czytane z `GARBAGE-CLASSIFICATION-3-2/data.yaml`: `BIODEGRADABLE`, `CARDBOARD`, `GLASS`, `METAL`, `PAPER`, `PLASTIC`.

Notebook zaklada, ze jest uruchamiany lokalnie na laptopie z dostepem do kamery. Domyslnie uzywa `cv2.VideoCapture(0)`, czyli standardowego indeksu wbudowanej kamery laptopa. Jesli Jupyter dziala na zdalnym serwerze, OpenCV bedzie probowal otworzyc kamere serwera, a nie kamere laptopa w przegladarce.

## 1. Instalacja zaleznosci

Uruchom ponizsza komorke tylko wtedy, gdy importy w kolejnym kroku nie dzialaja. Po instalacji najlepiej zrestartowac kernel notebooka.

In [ ]:
# Uruchom tylko raz, jesli brakuje bibliotek.
# Ta komenda korzysta z requirements.txt, gdzie PyTorch jest ustawiony na CPU-only,
# zeby nie pobierac duzych paczek CUDA na laptopie.
%pip install --timeout 120 -r requirements.txt


## 2. Lokalny custom dataset

In [1]:
from pathlib import Path

dataset_dir = Path("GARBAGE-CLASSIFICATION-3-2")
dataset_yaml = dataset_dir / "data.yaml"

if not dataset_yaml.exists():
    raise FileNotFoundError(f"Nie znaleziono lokalnego datasetu: {dataset_yaml.resolve()}")

print(f"Dataset OK: {dataset_dir.resolve()}")
print(f"Metadata: {dataset_yaml.resolve()}")


Dataset OK: /home/mikolaj/work/yolo-vs-rfdetr/GARBAGE-CLASSIFICATION-3-2
Metadata: /home/mikolaj/work/yolo-vs-rfdetr/GARBAGE-CLASSIFICATION-3-2/data.yaml


## 3. Importy, sciezki i konfiguracja

Ta komorka znajduje katalog projektu, sprawdza czy wagi i lokalny custom dataset istnieja, czyta nazwy klas z `GARBAGE-CLASSIFICATION-3-2/data.yaml` i wybiera najlepsze dostepne urzadzenie do inferencji.

In [2]:
import os
import time
from pathlib import Path

import yaml

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import cv2
import numpy as np
from PIL import Image
from IPython.display import clear_output, display

import torch
from ultralytics import YOLO
from rfdetr import RFDETRBase


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_weights = (candidate / "weights_yolo.pt").exists() and (candidate / "weights_rfdetr.pt").exists()
        has_dataset = (candidate / "GARBAGE-CLASSIFICATION-3-2" / "data.yaml").exists()
        if has_weights and has_dataset:
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()
YOLO_WEIGHTS = PROJECT_ROOT / "weights_yolo.pt"
RFDETR_WEIGHTS = PROJECT_ROOT / "weights_rfdetr.pt"
DATASET_DIR = PROJECT_ROOT / "GARBAGE-CLASSIFICATION-3-2"
DATASET_YAML = DATASET_DIR / "data.yaml"

CAMERA_INDEX = 0  # zwykle wbudowana kamera laptopa
CONFIDENCE = 0.35
DEMO_SECONDS = 30  # ustaw None, jesli demo ma dzialac az do przerwania komorki
DISPLAY_WIDTH = 900
MIRROR_CAMERA = True

if torch.cuda.is_available():
    DEVICE = "cuda:0"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

def load_dataset_class_names(dataset_yaml):
    with open(dataset_yaml, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    names = data.get("names")
    if isinstance(names, dict):
        return {int(idx): str(name) for idx, name in names.items()}
    if isinstance(names, list):
        return {idx: str(name) for idx, name in enumerate(names)}
    raise ValueError(f"Nie moge odczytac names z {dataset_yaml}")


missing = [str(path) for path in [YOLO_WEIGHTS, RFDETR_WEIGHTS, DATASET_YAML] if not path.exists()]
if missing:
    raise FileNotFoundError("Brakuje plikow wag: " + ", ".join(missing))

print(f"Project root: {PROJECT_ROOT}")
print(f"Custom dataset: {DATASET_DIR}")
DATASET_CLASS_NAMES = load_dataset_class_names(DATASET_YAML)
print(f"Classes: {DATASET_CLASS_NAMES}")
print(f"YOLO weights: {YOLO_WEIGHTS}")
print(f"RF-DETR weights: {RFDETR_WEIGHTS}")
print(f"Device: {DEVICE}")


/home/mikolaj/work/yolo-vs-rfdetr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /home/mikolaj/work/yolo-vs-rfdetr
Custom dataset: /home/mikolaj/work/yolo-vs-rfdetr/GARBAGE-CLASSIFICATION-3-2
Classes: {0: 'BIODEGRADABLE', 1: 'CARDBOARD', 2: 'GLASS', 3: 'METAL', 4: 'PAPER', 5: 'PLASTIC'}
YOLO weights: /home/mikolaj/work/yolo-vs-rfdetr/weights_yolo.pt
RF-DETR weights: /home/mikolaj/work/yolo-vs-rfdetr/weights_rfdetr.pt
Device: cpu


## 4. Funkcje pomocnicze do kamery i rysowania detekcji

Wspolny kod ponizej jest uzywany przez oba dema. Wyswietlanie odbywa sie bez `cv2.imshow`, dzieki czemu dziala w notebooku.

In [3]:
def to_numpy_1d(values, dtype=None):
    if values is None:
        return None
    if hasattr(values, "detach"):
        values = values.detach().cpu().numpy()
    arr = np.asarray(values)
    if dtype is not None:
        arr = arr.astype(dtype)
    return arr.reshape(-1)


def normalize_names(names):
    if names is None:
        return {}
    if isinstance(names, dict):
        return names
    return {idx: name for idx, name in enumerate(names)}


def draw_detections(frame_bgr, boxes, scores=None, class_ids=None, names=None, color=(0, 255, 0)):
    annotated = frame_bgr.copy()
    h, w = annotated.shape[:2]
    boxes = np.asarray(boxes if boxes is not None else [], dtype=np.float32).reshape(-1, 4)
    scores = to_numpy_1d(scores, np.float32)
    class_ids = to_numpy_1d(class_ids, np.int32)
    names = normalize_names(names)

    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box
        x1 = int(np.clip(x1, 0, w - 1))
        y1 = int(np.clip(y1, 0, h - 1))
        x2 = int(np.clip(x2, 0, w - 1))
        y2 = int(np.clip(y2, 0, h - 1))

        if x2 <= x1 or y2 <= y1:
            continue

        label_parts = []
        if class_ids is not None and idx < len(class_ids):
            cls_id = int(class_ids[idx])
            label_parts.append(str(names.get(cls_id, cls_id)))
        if scores is not None and idx < len(scores):
            label_parts.append(f"{float(scores[idx]):.2f}")
        label = " ".join(label_parts)

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        if label:
            (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            top = max(y1 - label_h - baseline - 6, 0)
            cv2.rectangle(annotated, (x1, top), (min(x1 + label_w + 8, w - 1), top + label_h + baseline + 6), color, -1)
            cv2.putText(annotated, label, (x1 + 4, top + label_h + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2, cv2.LINE_AA)

    return annotated


def frame_to_display_image(frame_bgr, display_width=DISPLAY_WIDTH):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(frame_rgb)
    if display_width is not None and image.width > display_width:
        new_height = int(image.height * display_width / image.width)
        image = image.resize((display_width, new_height))
    return image


def open_camera(camera_index=CAMERA_INDEX, width=1280, height=720):
    cap = cv2.VideoCapture(camera_index)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(
            f"Nie udalo sie otworzyc kamery laptopa index={camera_index}. "
            "Sprawdz uprawnienia systemowe albo uruchom probe_camera_indices(), zeby znalezc poprawny indeks."
        )
    return cap


def probe_camera_indices(max_index=4):
    """Opcjonalny test indeksow kamer. Kamera laptopa najczesciej jest pod indeksem 0."""
    found = []
    for idx in range(max_index + 1):
        cap = cv2.VideoCapture(idx)
        opened = cap.isOpened()
        grabbed = False
        if opened:
            grabbed, _ = cap.read()
        cap.release()
        if opened:
            found.append((idx, grabbed))

    if not found:
        print("Nie znaleziono zadnej kamery przez OpenCV.")
    else:
        for idx, grabbed in found:
            status = "OK" if grabbed else "otwarta, ale bez klatki"
            print(f"camera index {idx}: {status}")
        print("Ustaw CAMERA_INDEX na indeks wbudowanej kamery laptopa, zwykle 0.")


def run_live_demo(title, predict_frame_fn, seconds=DEMO_SECONDS, camera_index=CAMERA_INDEX, mirror=MIRROR_CAMERA):
    cap = open_camera(camera_index)
    start = time.perf_counter()
    frame_count = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError("Nie udalo sie odczytac klatki z kamery.")

            if mirror:
                frame = cv2.flip(frame, 1)

            annotated = predict_frame_fn(frame)
            frame_count += 1
            elapsed = max(time.perf_counter() - start, 1e-6)
            fps = frame_count / elapsed

            cv2.putText(annotated, f"{title} | FPS: {fps:.1f}", (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30, 30, 30), 4, cv2.LINE_AA)
            cv2.putText(annotated, f"{title} | FPS: {fps:.1f}", (12, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

            clear_output(wait=True)
            display(frame_to_display_image(annotated))

            if seconds is not None and elapsed >= seconds:
                break

    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
        clear_output(wait=True)
        print(f"{title}: zakonczono po {frame_count} klatkach.")


def show_camera_preview(seconds=5, camera_index=CAMERA_INDEX):
    def identity(frame):
        return frame

    run_live_demo("Camera preview", identity, seconds=seconds, camera_index=camera_index)


## 5. Test kamery

Najpierw sprawdz, czy notebook widzi wbudowana kamere laptopa. Domyslnie jest to `CAMERA_INDEX = 0`. Jesli preview nie dziala, odkomentuj `probe_camera_indices()` ponizej, znajdz poprawny indeks, zmien `CAMERA_INDEX` w konfiguracji i uruchom komorki od konfiguracji ponownie.

In [4]:
# Opcjonalnie, jesli kamera laptopa nie jest pod indeksem 0:
# probe_camera_indices(max_index=4)

show_camera_preview(seconds=5)


Camera preview: zakonczono po 16 klatkach.


## 6. Demo YOLO na zywo

Ta czesc laduje `weights_yolo.pt` i uruchamia detekcje z kamery. Etykiety klas pochodza z lokalnego `GARBAGE-CLASSIFICATION-3-2/data.yaml`. Domyslnie demo dziala 30 sekund. Zmien `DEMO_SECONDS = None` w konfiguracji, jesli ma dzialac do recznego przerwania komorki.

In [8]:
yolo_model = YOLO(str(YOLO_WEIGHTS))
yolo_names = DATASET_CLASS_NAMES or normalize_names(getattr(yolo_model, "names", None))
print("YOLO classes:", yolo_names)


def predict_yolo_frame(frame_bgr):
    result = yolo_model.predict(frame_bgr, conf=CONFIDENCE, device=DEVICE, verbose=False)[0]
    if result.boxes is None or result.boxes.xyxy is None:
        return frame_bgr.copy()

    boxes = result.boxes.xyxy.detach().cpu().numpy()
    scores = result.boxes.conf.detach().cpu().numpy() if result.boxes.conf is not None else None
    class_ids = result.boxes.cls.detach().cpu().numpy().astype(int) if result.boxes.cls is not None else None
    return draw_detections(frame_bgr, boxes, scores, class_ids, yolo_names, color=(255, 120, 40))


run_live_demo("YOLO", predict_yolo_frame)


YOLO: zakonczono po 176 klatkach.


## 7. Demo RF-DETR na zywo

Ta czesc laduje `weights_rfdetr.pt` jako checkpoint wytrenowany na customowym datasecie `GARBAGE-CLASSIFICATION-3-2` i uruchamia detekcje z kamery. Etykiety klas pochodza z lokalnego `data.yaml`, a konfiguracja architektury RF-DETR jest czytana z checkpointu. Pierwsze uruchomienie moze potrwac dluzej, bo model inicjalizuje graf inferencji.

In [9]:
def load_rfdetr_config_from_checkpoint(checkpoint_path):
    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")

    args = checkpoint.get("args", None) if isinstance(checkpoint, dict) else None
    args_dict = vars(args) if hasattr(args, "__dict__") else {}

    architecture_keys = [
        "encoder",
        "hidden_dim",
        "patch_size",
        "num_windows",
        "dec_layers",
        "sa_nheads",
        "ca_nheads",
        "dec_n_points",
        "num_queries",
        "num_select",
        "projector_scale",
        "out_feature_indexes",
        "resolution",
        "positional_encoding_size",
        "num_classes",
    ]

    config = {key: args_dict[key] for key in architecture_keys if key in args_dict}
    config["pretrain_weights"] = str(checkpoint_path)
    config["device"] = DEVICE
    return config, args_dict


def names_from_rfdetr_args(args_dict):
    if DATASET_CLASS_NAMES:
        return DATASET_CLASS_NAMES

    class_names = args_dict.get("class_names") or []
    if not class_names:
        num_classes = int(args_dict.get("num_classes") or 0)
        return {idx: str(idx) for idx in range(num_classes)}

    return {idx: name for idx, name in enumerate(class_names)}


rfdetr_config, rfdetr_args = load_rfdetr_config_from_checkpoint(RFDETR_WEIGHTS)
rfdetr_names = names_from_rfdetr_args(rfdetr_args)
rfdetr_model = RFDETRBase(**rfdetr_config)

print("RF-DETR config loaded from checkpoint:")
for key in ["patch_size", "num_windows", "dec_layers", "resolution", "positional_encoding_size", "num_classes"]:
    print(f"  {key}: {rfdetr_config.get(key)}")
print("Classes:", rfdetr_names)


def extract_rfdetr_output(detections):
    boxes = getattr(detections, "xyxy", None)
    scores = None
    class_ids = None

    for attr in ["confidence", "confidences", "scores", "score"]:
        if hasattr(detections, attr):
            scores = getattr(detections, attr)
            if scores is not None:
                break

    for attr in ["class_id", "class_ids", "classes"]:
        if hasattr(detections, attr):
            class_ids = getattr(detections, attr)
            if class_ids is not None:
                break

    return boxes, scores, class_ids


def predict_rfdetr_frame(frame_bgr):
    image_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(image_rgb)

    try:
        detections = rfdetr_model.predict(image, threshold=CONFIDENCE, include_source_image=False)
    except TypeError:
        detections = rfdetr_model.predict(image, threshold=CONFIDENCE)

    boxes, scores, class_ids = extract_rfdetr_output(detections)
    return draw_detections(frame_bgr, boxes, scores, class_ids, rfdetr_names, color=(0, 180, 255))


run_live_demo("RF-DETR", predict_rfdetr_frame)


RF-DETR: zakonczono po 49 klatkach.


## Wyniki treningu
Trening na 17k zdjęć z datasetu klasyfikacji śmieci, klasy to: 'Biodegradowalne', 'Kartony', 'Materiałowe', 'Szkło', 'Metal', 'Papier', 'Plastik'.

| Liczba parametrów | Model | mAP@50 | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| Około 3 mln | YOLOv11 (Nano) | 58.4% | 62.8% | 50.8% | 56.1% |
| Około 30 mln | RF-DETR (Small) | 66.7% | 68.6% | 62.3% | 65.3% |



## Nastepne kroki

1. Porównanie procesu treningu 3 modeli: Yolov8 medium, Yolov11 medium, RF-DETR small. Wykresy strat, mAP, precision, recall, F1 w czasie treningu.
3 modele, dla każdego w 3 rozdzielczościach 416, 640, 1280, trzeba pobrać w takiej rozdzielczości datasety z roboflow i potem zrobić trening, każdy model na takiej samej liczbie epok, podłączcie Comet ML, żebym mógł zrobić porównanie wykresów treningu.

2. Kwantyzacja i modyfikacja implementacji pod deploy na Nvidia Jetson Orin z kamerą w czasie rzeczywistym.
Tu trzeba będzie skwantyzowac do fp16 pewnie dla uproszczenia i zrobić jakaś prosta pętle inferencji z kamery


3. Porównanie wydajności modeli na Jetson Orin: FPS, zużycie pamięci, zużycie energii, dla różnych kwantyzacji oraz rozdzielczości wejściowej.
Zrobić kwantyzacje modeli i porównać wyndajnośc w zależności od kwantyzacji i rozdzielczoci wejściowe:

Coś tego typu:
| Model         | Format        | Typ wag       | Rozmiar (MB) | Śr. czas inferencji (ms) | Wymiary klatki | FPS   | Zużycie GPU (%) |
|---------------|---------------|---------------|--------------|--------------------------|----------------|-------|-----------------|
| YOLOv5n       | .engine       | float32       | 9.7          | 6.8                      | 640x640        | 147   | 29.35           |
| YOLOv5s       | .engine       | float32       | 34.7         | 13.0                     | 640x640        | 76    | 33.96           |
| YOLOv5m       | .engine       | float32       | 93.5         | 33.2                     | 640x640        | 30    | 47.94           |
| YOLOv5n       | .pt           | float32       | 3.8          | 9.7                      | 640x640        | 103   | 32.52           |
| YOLOv5s       | .pt           | float32       | 14.1         | 18.9                     | 640x640        | 52    | 38.67           |
| YOLOv5m       | .pt           | float32       | 40.8         | 46.7                     | 640x640        | 21    | 56.63           |
| YOLOv5n       | .engine       | float16       | 16.4         | 7.8                      | 640x640        | 128   | 27.59           |
| YOLOv5s       | .engine       | float16       | 50.4         | 16.5                     | 640x640        | 60    | 37.03           |
| YOLOv5m       | .engine       | float16       | 126.6        | 37.3                     | 640x640        | 26    | 50.71           |
| YOLOv5n       | .engine       | int8          | 16.5         | 7.8                      | 640x640        | 128   | 27.93           |
| YOLOv5s       | .engine       | int8          | 50.4         | 17.4                     | 640x640        | 57    | 37.53           |
| YOLOv5m       | .engine       | int8          | 126.5        | 37.9                     | 640x640        | 26    | 53.42           |
